# Intelligent Pre-Triage System v2.0

**Sistema Inteligente de Pré-Triagem Médica**

Sistema conversacional para pré-triagem utilizando LLMs locais via Ollama.

---

**AVISO ÉTICO:** Este sistema NÃO realiza diagnóstico médico. Ele apenas coleta sintomas, estrutura informações, converte linguagem leiga em termos médicos, gera anamnese estruturada e apresenta estatísticas históricas.

---

### Changelog v2.0
- Correção do erro `Expecting value: line 1 column 1 (char 0)`
- Uso do parâmetro `format` do Ollama para forçar JSON estruturado
- System message dedicada para controlar comportamento do modelo
- Sanitização robusta da resposta com fallback regex
- Validação de schema com Pydantic
- Retry automático com até 3 tentativas
- Temperature 0 para respostas determinísticas
- Tratamento de exceções detalhado com logging

## 1. Importações e Dependências

In [15]:
import json
import re
import logging
from typing import Optional

import pandas as pd
import ollama

# Pydantic para validação de schema
# Instalar se necessário: pip install pydantic
from pydantic import BaseModel, Field, ValidationError

## 2. Configuração de Logging

In [16]:
# Configurar logging para facilitar debug
# Em produção, altere o nível para logging.WARNING

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

logger = logging.getLogger("pre_triagem")

## 3. Configurações Globais

Centralizamos todas as configurações em um único local para facilitar ajustes.

In [17]:
# ============================================================
# CONFIGURAÇÕES DO SISTEMA
# ============================================================

# Modelo LLM a ser utilizado via Ollama.
# Opções recomendadas: "phi3", "llama3", "llama3.1"
MODELO = "phi3"

# Número máximo de tentativas ao chamar o modelo
MAX_TENTATIVAS = 3

# Número de exemplos few-shot do dataset
NUM_EXEMPLOS_FEW_SHOT = 3

# Opções do Ollama para controlar o comportamento do modelo
OPCOES_OLLAMA = {
    "temperature": 0.0,     # Determinístico: reduz variabilidade
    "top_p": 0.9,          # Nucleus sampling conservador
    "num_predict": 1024,   # Limite de tokens na resposta
    "seed": 42             # Semente fixa para reprodutibilidade
}

# Categoria padrão quando nenhum sintoma é reconhecido
CATEGORIA_PADRAO = 5

## 4. Schema de Validação com Pydantic

Definimos o schema esperado da resposta do modelo usando Pydantic. Isso garante:
- Validação automática de tipos
- Valores padrão para campos ausentes
- Serialização consistente para JSON
- Uso como parâmetro `format` do Ollama (structured outputs)

In [18]:
class AnamneseSchema(BaseModel):
    """Schema de validação para a resposta do modelo LLM."""

    queixa_principal: str = Field(
        default="Não informado",
        description="Queixa principal do paciente em termos médicos"
    )

    duracao: str = Field(
        default="Não informado",
        description="Duração dos sintomas relatados"
    )

    intensidade: str = Field(
        default="Não informado",
        description="Intensidade dos sintomas (leve, moderada, intensa)"
    )

    sintomas_associados: list[str] = Field(
        default_factory=list,
        description="Lista de sintomas associados em termos médicos"
    )

    historico_medico: list[str] = Field(
        default_factory=list,
        description="Histórico médico relevante mencionado"
    )

    anamnese: str = Field(
        default="Não informado",
        description="Texto completo da anamnese estruturada"
    )


# Exibir o schema JSON que será enviado ao Ollama
logger.info("Schema Pydantic carregado com sucesso.")
print(json.dumps(AnamneseSchema.model_json_schema(), indent=2, ensure_ascii=False))

2026-05-28 18:44:45,861 [INFO] Schema Pydantic carregado com sucesso.


{
  "description": "Schema de validação para a resposta do modelo LLM.",
  "properties": {
    "queixa_principal": {
      "default": "Não informado",
      "description": "Queixa principal do paciente em termos médicos",
      "title": "Queixa Principal",
      "type": "string"
    },
    "duracao": {
      "default": "Não informado",
      "description": "Duração dos sintomas relatados",
      "title": "Duracao",
      "type": "string"
    },
    "intensidade": {
      "default": "Não informado",
      "description": "Intensidade dos sintomas (leve, moderada, intensa)",
      "title": "Intensidade",
      "type": "string"
    },
    "sintomas_associados": {
      "description": "Lista de sintomas associados em termos médicos",
      "items": {
        "type": "string"
      },
      "title": "Sintomas Associados",
      "type": "array"
    },
    "historico_medico": {
      "description": "Histórico médico relevante mencionado",
      "items": {
        "type": "string"
      },
    

## 5. Carregamento dos Datasets

In [19]:
# ============================================================
# CARREGAMENTO DOS DATASETS
# ============================================================

# 1. dataset_triagem.csv
#    Exemplos clínicos para few-shot prompting.
#
# 2. estatisticas_por_categoria.csv
#    Estatísticas históricas por categoria de triagem.
#
# 3. durchlauf1.csv
#    Registros históricos complementares.

try:
    dataset_triagem = pd.read_csv(
        "dataset_triagem.csv",
        sep=",",
        encoding="latin1"
    )

    estatisticas = pd.read_csv(
        "estatisticas_por_categoria.csv",
        sep=",",
        encoding="latin1"
    )

    durchlauf = pd.read_csv(
        "durchlauf1.csv",
        sep=";",
        encoding="latin1"
    )

    # Limitar exemplos few-shot
    dataset_triagem = dataset_triagem.head(NUM_EXEMPLOS_FEW_SHOT)

    print("=" * 60)
    print("DATASETS CARREGADOS COM SUCESSO")
    print("=" * 60)
    print(f"\nDataset triagem:  {dataset_triagem.shape}")
    print(f"Estatísticas:     {estatisticas.shape}")
    print(f"Durchlauf:        {durchlauf.shape}")

    logger.info("Todos os datasets foram carregados.")

except FileNotFoundError as e:
    logger.error(f"Arquivo não encontrado: {e}")
    raise

except Exception as e:
    logger.error(f"Erro ao carregar datasets: {e}")
    raise

2026-05-28 18:44:45,928 [INFO] Todos os datasets foram carregados.


DATASETS CARREGADOS COM SUCESSO

Dataset triagem:  (3, 7)
Estatísticas:     (5, 6)
Durchlauf:        (29564, 9)


## 6. Mapa de Categorias Estatísticas

Relaciona sintomas identificados na anamnese com categorias estatísticas históricas do dataset.

In [20]:
MAPA_CATEGORIAS = {
    # Categoria 2: Sintomas cardiorrespiratórios
    "dor torácica": 2,
    "dor no peito": 2,
    "dispneia": 2,
    "falta de ar": 2,
    "taquicardia": 2,
    "palpitação": 2,

    # Categoria 3: Sintomas gastrointestinais
    "êmese": 3,
    "vomito": 3,
    "vômito": 3,
    "náusea": 3,
    "diarreia": 3,
    "dor abdominal": 3,

    # Categoria 4: Sintomas gerais / infecciosos
    "febre": 4,
    "cefaleia": 4,
    "tosse": 4,
    "dor de cabeça": 4,
    "calafrios": 4,
    "mialgia": 4,
    "dor de garganta": 4,
    "odinofagia": 4,
    "coriza": 4
}

## 7. Funções Auxiliares

### 7.1 Identificação de Categoria Estatística

In [21]:
def identificar_categoria(texto: str) -> int:
    """
    Identifica a categoria estatística relacionada
    aos sintomas encontrados no texto da anamnese.

    Parâmetros:
        texto: Texto da anamnese ou queixa do paciente.

    Retorna:
        Número da categoria (2, 3, 4 ou 5 como padrão).
    """
    texto_lower = texto.lower()

    for sintoma, categoria in MAPA_CATEGORIAS.items():
        if sintoma in texto_lower:
            logger.info(
                f"Sintoma '{sintoma}' identificado -> Categoria {categoria}"
            )
            return categoria

    logger.info(
        f"Nenhum sintoma mapeado encontrado. Usando categoria padrão {CATEGORIA_PADRAO}."
    )
    return CATEGORIA_PADRAO

### 7.2 Consulta de Estatísticas Históricas

In [22]:
def buscar_estatisticas(categoria: int) -> Optional[dict]:
    """
    Busca estatísticas históricas relacionadas
    à categoria identificada.

    Parâmetros:
        categoria: Número da categoria estatística.

    Retorna:
        Dicionário com tempos (médio, mediano, mínimo, máximo)
        ou None se a categoria não for encontrada.
    """
    linha = estatisticas[
        estatisticas["triagestufe"] == categoria
    ]

    if len(linha) == 0:
        logger.warning(
            f"Categoria {categoria} não encontrada no dataset de estatísticas."
        )
        return None

    linha = linha.iloc[0]

    return {
        "tempo_medio": linha["mean"],
        "tempo_mediano": linha["median"],
        "tempo_minimo": linha["min"],
        "tempo_maximo": linha["max"]
    }

## 8. Sanitização e Extração de JSON

Esta é a **correção principal** do erro `Expecting value: line 1 column 1 (char 0)`.

O problema ocorre porque modelos LLM frequentemente retornam JSON envolvido em:
- Blocos markdown (` ```json ... ``` `)
- Texto explicativo antes ou depois do JSON
- Valores inválidos como `NaN`, `None`, `undefined`

A função `extrair_json_da_resposta()` implementa uma cadeia de fallbacks:

1. Tenta `json.loads()` direto
2. Remove blocos markdown e tenta novamente
3. Extrai JSON via regex (busca `{ ... }`)
4. Corrige valores inválidos (`NaN`, `null` sem aspas, etc.)

In [23]:
def sanitizar_json_texto(texto: str) -> str:
    """
    Corrige problemas comuns em JSON gerado por LLMs.

    Correções aplicadas:
        - Substitui NaN (sem aspas) por "Não informado"
        - Substitui None/null (sem aspas) por "Não informado"
        - Substitui undefined por "Não informado"
        - Remove vírgulas antes de ] ou }
        - Remove caracteres de controle
    """
    # Substituir NaN (não entre aspas) por string válida
    texto = re.sub(
        r'(?<!["\w])NaN(?!["\w])',
        '"Não informado"',
        texto
    )

    # Substituir None (não entre aspas) por string válida
    texto = re.sub(
        r'(?<!["\w])None(?!["\w])',
        '"Não informado"',
        texto
    )

    # Substituir undefined por string válida
    texto = re.sub(
        r'(?<!["\w])undefined(?!["\w])',
        '"Não informado"',
        texto
    )

    # Remover trailing commas (vírgula antes de ] ou })
    texto = re.sub(r',\s*([}\]])', r'\1', texto)

    # Remover caracteres de controle (exceto \n, \r, \t)
    texto = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', texto)

    return texto


def extrair_json_da_resposta(resposta_bruta: str) -> dict:
    """
    Extrai e valida JSON da resposta do modelo LLM.

    Implementa uma cadeia de fallbacks para lidar com
    respostas que contêm markdown, texto extra ou
    valores inválidos.

    Parâmetros:
        resposta_bruta: String retornada pelo modelo.

    Retorna:
        Dicionário Python com os dados extraídos.

    Levanta:
        ValueError: Se não for possível extrair JSON válido.
    """
    if not resposta_bruta or not resposta_bruta.strip():
        raise ValueError("Resposta do modelo está vazia.")

    texto = resposta_bruta.strip()

    # ---- TENTATIVA 1: json.loads() direto ----
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        logger.info("Tentativa 1 falhou (json.loads direto). Tentando limpeza...")

    # ---- TENTATIVA 2: Remover blocos markdown ```json ... ``` ----
    padrao_markdown = re.search(
        r'```(?:json)?\s*\n?(.*?)\n?\s*```',
        texto,
        re.DOTALL
    )

    if padrao_markdown:
        conteudo_bloco = padrao_markdown.group(1).strip()
        conteudo_bloco = sanitizar_json_texto(conteudo_bloco)

        try:
            return json.loads(conteudo_bloco)
        except json.JSONDecodeError:
            logger.info("Tentativa 2 falhou (bloco markdown). Tentando regex...")

    # ---- TENTATIVA 3: Extrair primeiro objeto JSON { ... } via regex ----
    padrao_json = re.search(
        r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}',
        texto,
        re.DOTALL
    )

    if padrao_json:
        candidato = padrao_json.group(0).strip()
        candidato = sanitizar_json_texto(candidato)

        try:
            return json.loads(candidato)
        except json.JSONDecodeError:
            logger.info("Tentativa 3 falhou (regex de objeto JSON).")

    # ---- TENTATIVA 4: Sanitizar texto completo e tentar novamente ----
    texto_sanitizado = sanitizar_json_texto(texto)

    try:
        return json.loads(texto_sanitizado)
    except json.JSONDecodeError as e:
        logger.error(f"Todas as tentativas de extração falharam.")
        logger.error(f"Resposta bruta (primeiros 500 chars): {texto[:500]}")
        raise ValueError(
            f"Não foi possível extrair JSON válido da resposta do modelo.\n"
            f"Erro: {e}\n"
            f"Resposta recebida (início): {texto[:200]}..."
        )

## 9. Validação com Pydantic

Após extrair o JSON, validamos a estrutura usando o schema Pydantic. Isso garante que todos os campos existam e tenham os tipos corretos.

In [24]:
def validar_anamnese(dados_dict: dict) -> AnamneseSchema:
    """
    Valida o dicionário extraído contra o schema Pydantic.

    Campos ausentes recebem valores padrão.
    Campos com tipos incorretos são convertidos quando possível.

    Parâmetros:
        dados_dict: Dicionário extraído da resposta do modelo.

    Retorna:
        Instância validada de AnamneseSchema.
    """
    try:
        anamnese = AnamneseSchema(**dados_dict)
        logger.info("Validação Pydantic: OK")
        return anamnese

    except ValidationError as e:
        logger.warning(f"Validação Pydantic falhou: {e}")
        logger.warning("Tentando correção automática dos campos...")

        # Correção automática: converter campos problemáticos
        dados_corrigidos = {}

        for campo in AnamneseSchema.model_fields:
            valor = dados_dict.get(campo)

            if valor is None or valor == "NaN" or valor == "null":
                # Deixar o Pydantic usar o valor padrão
                continue

            if campo in ("sintomas_associados", "historico_medico"):
                # Garantir que seja lista
                if isinstance(valor, str):
                    valor = [item.strip() for item in valor.split(",") if item.strip()]
                elif not isinstance(valor, list):
                    valor = []
            else:
                # Garantir que seja string
                if not isinstance(valor, str):
                    valor = str(valor)

            dados_corrigidos[campo] = valor

        return AnamneseSchema(**dados_corrigidos)

## 10. Construção do Prompt

O prompt foi reestruturado com as seguintes melhorias:

- **System message** separada para definir o papel do modelo
- **Instruções negativas explícitas** (sem markdown, sem explicações)
- **Schema JSON** incluído no prompt para grounding
- **Exemplos few-shot** formatados de forma limpa
- **Delimitadores claros** entre seções

In [25]:
# System message: define o comportamento global do modelo
SYSTEM_MESSAGE = """Você é um assistente de pré-triagem médica que responde EXCLUSIVAMENTE em JSON válido.

REGRAS ABSOLUTAS:
1. Responda APENAS com um objeto JSON válido. Nada mais.
2. NÃO inclua blocos de código markdown (```json).
3. NÃO inclua explicações, comentários ou texto adicional.
4. NÃO realize diagnóstico médico.
5. NÃO sugira tratamentos ou medicamentos.
6. NÃO classifique risco clínico.
7. Apenas organize e estruture as informações do paciente.
8. Converta linguagem leiga para termos médicos quando apropriado.
9. Se uma informação não foi fornecida, use "Não informado".
10. Sua resposta deve começar com { e terminar com }."""


def construir_prompt(texto_paciente: str) -> str:
    """
    Constrói o prompt do usuário com exemplos few-shot
    e o caso atual do paciente.

    Parâmetros:
        texto_paciente: Descrição dos sintomas pelo paciente.

    Retorna:
        String do prompt formatado.
    """
    # Cabeçalho com schema esperado
    prompt = """Analise o relato do paciente e retorne APENAS um JSON válido no formato abaixo.

Schema esperado:
{
    "queixa_principal": "string - queixa principal em termos médicos",
    "duracao": "string - duração dos sintomas",
    "intensidade": "string - leve, moderada ou intensa",
    "sintomas_associados": ["lista de sintomas em termos médicos"],
    "historico_medico": ["lista de condições prévias mencionadas"],
    "anamnese": "string - texto completo da anamnese estruturada"
}

--- EXEMPLOS DE REFERÊNCIA ---
"""

    # Inserir exemplos few-shot do dataset
    for idx, linha in dataset_triagem.iterrows():

        sintomas_associados = []
        historico_medico = []

        if pd.notna(linha.get("sintomas_associados", None)):
            sintomas_associados = [
                item.strip()
                for item in str(linha["sintomas_associados"]).split(";")
                if item.strip()
            ]

        if pd.notna(linha.get("historico_medico", None)):
            historico_medico = [
                item.strip()
                for item in str(linha["historico_medico"]).split(";")
                if item.strip()
            ]

        exemplo_saida = {
            "queixa_principal": str(linha.get("queixa_principal", "Não informado")),
            "duracao": str(linha.get("duracao", "Não informado")),
            "intensidade": str(linha.get("intensidade", "Não informado")),
            "sintomas_associados": sintomas_associados,
            "historico_medico": historico_medico,
            "anamnese": str(linha.get("anamnese", "Não informado"))
        }

        prompt += f"""
Exemplo {idx + 1}:
Paciente: {linha['entrada']}
Resposta: {json.dumps(exemplo_saida, ensure_ascii=False)}
"""

    # Caso atual
    prompt += f"""
--- CASO ATUAL ---

Paciente: {texto_paciente}
Resposta:"""

    return prompt

## 11. Geração da Anamnese com Retry

A função principal que chama o modelo Ollama com:
- **System message** para controlar comportamento
- **Parâmetro `format`** com schema JSON (structured outputs do Ollama)
- **Temperature 0** para respostas determinísticas
- **Retry automático** com até 3 tentativas
- **Sanitização e validação** da resposta

In [26]:
def gerar_anamnese(texto_paciente: str) -> AnamneseSchema:
    """
    Gera uma anamnese estruturada a partir do relato do paciente.

    Pipeline completo:
        1. Constrói o prompt com exemplos few-shot
        2. Envia ao modelo Ollama com structured output
        3. Extrai JSON da resposta (com fallbacks)
        4. Valida contra schema Pydantic
        5. Retenta até MAX_TENTATIVAS vezes em caso de falha

    Parâmetros:
        texto_paciente: Descrição dos sintomas pelo paciente.

    Retorna:
        Instância validada de AnamneseSchema.

    Levanta:
        RuntimeError: Se todas as tentativas falharem.
    """
    prompt = construir_prompt(texto_paciente)
    ultimo_erro = None

    for tentativa in range(1, MAX_TENTATIVAS + 1):

        logger.info(f"Tentativa {tentativa}/{MAX_TENTATIVAS}...")

        try:
            # Chamada ao modelo Ollama
            resposta = ollama.chat(
                model=MODELO,
                messages=[
                    {
                        "role": "system",
                        "content": SYSTEM_MESSAGE
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                format=AnamneseSchema.model_json_schema(),
                options=OPCOES_OLLAMA
            )

            resposta_bruta = resposta["message"]["content"]

            logger.info(
                f"Resposta recebida ({len(resposta_bruta)} caracteres)."
            )
            logger.debug(
                f"Resposta bruta: {resposta_bruta[:300]}"
            )

            # Extrair JSON (com cadeia de fallbacks)
            dados_dict = extrair_json_da_resposta(resposta_bruta)

            # Validar contra schema Pydantic
            anamnese_validada = validar_anamnese(dados_dict)

            logger.info("Anamnese gerada e validada com sucesso.")
            return anamnese_validada

        except (ValueError, ValidationError, json.JSONDecodeError) as e:
            ultimo_erro = e
            logger.warning(
                f"Tentativa {tentativa} falhou: {type(e).__name__}: {e}"
            )

            if tentativa < MAX_TENTATIVAS:
                logger.info("Retentando...")

        except Exception as e:
            # Erros de conexão com Ollama, modelo não encontrado, etc.
            ultimo_erro = e
            logger.error(
                f"Erro inesperado na tentativa {tentativa}: {type(e).__name__}: {e}"
            )
            break

    raise RuntimeError(
        f"Falha ao gerar anamnese após {MAX_TENTATIVAS} tentativas.\n"
        f"Último erro: {ultimo_erro}"
    )

## 12. Exibição dos Resultados

In [27]:
def exibir_anamnese(anamnese: AnamneseSchema) -> None:
    """
    Exibe a anamnese estruturada de forma organizada.
    """
    print("\n")
    print("=" * 60)
    print("ANAMNESE ESTRUTURADA")
    print("=" * 60)

    print(f"\nQueixa principal:")
    print(f"  {anamnese.queixa_principal}")

    print(f"\nDuração:")
    print(f"  {anamnese.duracao}")

    print(f"\nIntensidade:")
    print(f"  {anamnese.intensidade}")

    print(f"\nSintomas associados:")
    if not anamnese.sintomas_associados:
        print("  - Nenhum")
    else:
        for sintoma in anamnese.sintomas_associados:
            print(f"  - {sintoma}")

    print(f"\nHistórico médico:")
    if not anamnese.historico_medico:
        print("  - Nenhum")
    else:
        for item in anamnese.historico_medico:
            print(f"  - {item}")

    print(f"\nAnamnese:")
    print(f"  {anamnese.anamnese}")


def exibir_estatisticas(anamnese: AnamneseSchema) -> None:
    """
    Identifica categoria e exibe estatísticas históricas.
    """
    # Construir texto de busca
    texto_busca = anamnese.queixa_principal
    if anamnese.sintomas_associados:
        texto_busca += " " + " ".join(anamnese.sintomas_associados)

    categoria = identificar_categoria(texto_busca)
    estatisticas_cat = buscar_estatisticas(categoria)

    print("\n")
    print("=" * 60)
    print("DADOS ESTATÍSTICOS RELACIONADOS")
    print("=" * 60)

    print("""
    Os dados abaixo representam apenas
    estatísticas históricas relacionadas
    a sintomas semelhantes.

    Eles NÃO representam:
    - diagnóstico;
    - classificação médica;
    - decisão clínica.
    """)

    print(f"Categoria estatística: {categoria}")

    if estatisticas_cat is not None:
        print(f"\nTempo médio histórico:   {estatisticas_cat['tempo_medio']}")
        print(f"Tempo mediano histórico: {estatisticas_cat['tempo_mediano']}")
        print(f"Tempo mínimo histórico:  {estatisticas_cat['tempo_minimo']}")
        print(f"Tempo máximo histórico:  {estatisticas_cat['tempo_maximo']}")
    else:
        print("\nEstatísticas não disponíveis para esta categoria.")

## 13. Execução Principal

Fluxo completo do sistema com tratamento de erros robusto.

In [28]:
# ============================================================
# SISTEMA INTELIGENTE DE PRÉ-TRIAGEM
# ============================================================

print("\n")
print("=" * 60)
print("SISTEMA INTELIGENTE DE PRÉ-TRIAGEM")
print("=" * 60)

print("""
AVISO IMPORTANTE

Este sistema NÃO realiza diagnóstico,
NÃO substitui profissionais de saúde
e NÃO realiza classificação clínica.

O sistema apenas organiza informações
para apoio administrativo.
""")

entrada = input("\nDescreva seus sintomas:\n> ")

if not entrada.strip():
    print("\nNenhum sintoma informado. Encerrando.")
else:
    print("\nProcessando... Aguarde a resposta do modelo.")

    try:
        anamnese = gerar_anamnese(entrada)

        # Exibir JSON bruto validado
        print("\n")
        print("=" * 60)
        print("RESPOSTA DO MODELO (JSON VALIDADO)")
        print("=" * 60)
        print(json.dumps(
            anamnese.model_dump(),
            ensure_ascii=False,
            indent=4
        ))

        # Exibir anamnese formatada
        exibir_anamnese(anamnese)

        # Exibir estatísticas
        exibir_estatisticas(anamnese)

    except RuntimeError as e:
        print(f"\nErro: {e}")
        print("\nSugestões:")
        print("1. Verifique se o Ollama está em execução (ollama serve)")
        print(f"2. Verifique se o modelo '{MODELO}' está instalado (ollama pull {MODELO})")
        print("3. Tente reformular a descrição dos sintomas")
        print("4. Tente usar outro modelo (ex: llama3)")

    except Exception as e:
        print(f"\nErro inesperado: {type(e).__name__}: {e}")
        logger.exception("Erro não tratado no fluxo principal.")



SISTEMA INTELIGENTE DE PRÉ-TRIAGEM

AVISO IMPORTANTE

Este sistema NÃO realiza diagnóstico,
NÃO substitui profissionais de saúde
e NÃO realiza classificação clínica.

O sistema apenas organiza informações
para apoio administrativo.



2026-05-28 18:47:51,094 [INFO] Tentativa 1/3...



Processando... Aguarde a resposta do modelo.


2026-05-28 18:49:53,949 [INFO] HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-05-28 18:49:53,952 [INFO] Resposta recebida (298 caracteres).
2026-05-28 18:49:53,953 [INFO] Validação Pydantic: OK
2026-05-28 18:49:53,954 [INFO] Anamnese gerada e validada com sucesso.
2026-05-28 18:49:53,955 [INFO] Sintoma 'dor torácica' identificado -> Categoria 2




RESPOSTA DO MODELO (JSON VALIDADO)
{
    "queixa_principal": "dor torácica",
    "duracao": "desconhecida",
    "intensidade": "forte",
    "sintomas_associados": [
        "dispneia"
    ],
    "historico_medico": [],
    "anamnese": "Paciente relata dor torácica intensa associada à dispneia sem informações adicionais sobre histórico médico."
}


ANAMNESE ESTRUTURADA

Queixa principal:
  dor torácica

Duração:
  desconhecida

Intensidade:
  forte

Sintomas associados:
  - dispneia

Histórico médico:
  - Nenhum

Anamnese:
  Paciente relata dor torácica intensa associada à dispneia sem informações adicionais sobre histórico médico.


DADOS ESTATÍSTICOS RELACIONADOS

    Os dados abaixo representam apenas
    estatísticas históricas relacionadas
    a sintomas semelhantes.

    Eles NÃO representam:
    - diagnóstico;
    - classificação médica;
    - decisão clínica.
    
Categoria estatística: 2

Tempo médio histórico:   879.3899848254931
Tempo mediano histórico: 434.0
Tempo mínimo h